# LLM Observability & Cost Management

**Track:** Production Engineering  
**Toolchain:** Langfuse · LangSmith · Token Budgeting · Cost Dashboards  
**Objective:** Learn how to monitor, debug, and control costs for LLM applications in production.

---

## 🧠 Why LLM Observability Is Different

Traditional software monitoring: track latency, errors, throughput.  
LLM monitoring: track ALL of that + **tokens, costs, quality, hallucinations, and prompt versions.**

| Traditional Monitoring | LLM Observability |
|----------------------|-----------------|
| Response time | Response time + time-to-first-token |
| Error rate | Error rate + hallucination rate |
| CPU/memory usage | GPU utilization + VRAM usage |
| Request count | Token count (input + output) |
| Server cost | LLM API cost per request |
| N/A | Prompt version tracking |
| N/A | Conversation quality scoring |

### The LLM Observability Stack

```
Level 4: Business Metrics     ← User satisfaction, task completion rate
Level 3: Quality Metrics      ← Faithfulness, relevance, hallucination rate
Level 2: Token & Cost         ← Tokens per request, cost per user, budget alerts
Level 1: Infrastructure       ← Latency, throughput, GPU utilization, errors
```

### Observability Tools (2026)

| Tool | Strengths | Open Source | Best For |
|------|----------|------------|----------|
| **Langfuse** | Tracing, prompt management, scoring | ✅ Yes | All-around observability |
| **LangSmith** | Deep LangChain integration, datasets | ❌ No | LangChain-based apps |
| **Helicone** | Simple proxy, cost tracking | ❌ No (free tier) | Quick cost monitoring |
| **Arize Phoenix** | Embedding analysis, notebooks | ✅ Yes | Research & debugging |
| **W&B Weave** | Tracing + experiment tracking | ❌ No | Teams already using W&B |

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You cannot optimize what you cannot trace. Seniors instrument every LLM call, tracing the exact sequence, token count, latency, and cost of sub-components to identify bottlenecks and stop runaway spending.

**Mental Model:** LLM Observability (Langfuse) is a highly detailed itemized grocery receipt. It shows you exactly how much 'Research Retrieval' and 'Context Engineering' contributed to the final $0.05 bill of the response.

**Common Junior Pitfall:** Deploying without token budgets. A runaway recursive agent loop or a malicious user can rack up thousands of dollars in API fees overnight. Strict per-user daily budgets are critical.

---


In [ ]:
# Cell 1 — Install
!pip install -q pydantic numpy

## 1 · Request Tracing: Follow Every Token

### 🤔 What is a Trace?

A **trace** records everything that happens during a single user request:

```
Trace: "user asked about Q4 revenue"
  ├── Span: Input guardrails     (2ms, passed)
  ├── Span: RAG retrieval        (150ms, 5 chunks found)
  ├── Span: Prompt assembly      (1ms, 1,200 tokens)
  ├── Span: LLM call             (2,100ms, 350 output tokens, $0.012)
  ├── Span: Output guardrails    (5ms, passed)
  └── Span: Response delivery    (3ms)
  Total: 2,261ms | Cost: $0.012 | Quality: 0.92
```

**Analogy:** A trace is like a receipt that shows every step in your order — from kitchen prep to delivery — with costs and timing.

In [ ]:
# Cell 2 — LLM request tracing system
import time, json, uuid
from datetime import datetime

class LLMTrace:
    """Production-style LLM request tracing."""
    
    def __init__(self, user_id, session_id=None):
        self.trace_id = str(uuid.uuid4())[:8]
        self.user_id = user_id
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.spans = []
        self.start_time = time.time()
        self.metadata = {}
    
    def add_span(self, name, duration_ms, tokens_in=0, tokens_out=0, cost=0, status='ok', metadata=None):
        span = {
            'name': name, 'duration_ms': duration_ms, 'status': status,
            'tokens_in': tokens_in, 'tokens_out': tokens_out, 'cost': cost,
            'metadata': metadata or {}
        }
        self.spans.append(span)
        return span
    
    def summary(self):
        total_ms = sum(s['duration_ms'] for s in self.spans)
        total_tokens_in = sum(s['tokens_in'] for s in self.spans)
        total_tokens_out = sum(s['tokens_out'] for s in self.spans)
        total_cost = sum(s['cost'] for s in self.spans)
        return {
            'trace_id': self.trace_id, 'user_id': self.user_id,
            'total_ms': total_ms, 'tokens_in': total_tokens_in,
            'tokens_out': total_tokens_out, 'cost': f'${total_cost:.4f}',
            'spans': len(self.spans),
        }

class ObservabilityDashboard:
    """Aggregates traces into a monitoring dashboard."""
    
    def __init__(self):
        self.traces = []
    
    def record(self, trace):
        self.traces.append(trace)
    
    def report(self):
        if not self.traces:
            return {}
        summaries = [t.summary() for t in self.traces]
        total_cost = sum(float(s['cost'][1:]) for s in summaries)
        avg_latency = sum(s['total_ms'] for s in summaries) / len(summaries)
        total_tokens = sum(s['tokens_in'] + s['tokens_out'] for s in summaries)
        
        # Per-user cost
        user_costs = {}
        for s in summaries:
            user_costs[s['user_id']] = user_costs.get(s['user_id'], 0) + float(s['cost'][1:])
        
        return {
            'total_requests': len(self.traces),
            'total_cost': f'${total_cost:.4f}',
            'avg_latency_ms': f'{avg_latency:.0f}',
            'total_tokens': total_tokens,
            'cost_per_request': f'${total_cost/len(self.traces):.4f}',
            'top_users_by_cost': dict(sorted(user_costs.items(), key=lambda x: x[1], reverse=True)[:3]),
        }

# Simulate production traffic
import random
random.seed(42)

dashboard = ObservabilityDashboard()

for i in range(20):
    user = random.choice(['user_alice', 'user_bob', 'user_charlie', 'user_dave'])
    trace = LLMTrace(user_id=user)
    
    trace.add_span('input_guardrails', duration_ms=3, status='ok')
    trace.add_span('rag_retrieval', duration_ms=random.randint(80, 200), metadata={'chunks': random.randint(3, 7)})
    tokens_in = random.randint(500, 2000)
    tokens_out = random.randint(100, 500)
    cost = tokens_in * 0.000005 + tokens_out * 0.000015  # GPT-4o pricing approx
    trace.add_span('llm_call', duration_ms=random.randint(800, 3000),
                   tokens_in=tokens_in, tokens_out=tokens_out, cost=cost,
                   metadata={'model': 'gpt-4o', 'temperature': 0.1})
    trace.add_span('output_guardrails', duration_ms=5, status='ok')
    
    dashboard.record(trace)

# Show a sample trace
sample = dashboard.traces[0]
print('📊 Sample Trace')
print(f'  Trace ID: {sample.trace_id} | User: {sample.user_id}')
for span in sample.spans:
    tokens = f' [{span["tokens_in"]}→{span["tokens_out"]} tokens]' if span['tokens_in'] else ''
    cost = f' ${span["cost"]:.4f}' if span['cost'] else ''
    print(f'  ├── {span["name"]:<20} {span["duration_ms"]:>5}ms{tokens}{cost}')
s = sample.summary()
print(f'  └── Total: {s["total_ms"]}ms | {s["cost"]}')

# Dashboard report
print(f'\n📈 Dashboard Report ({dashboard.report()["total_requests"]} requests)')
print('=' * 50)
for k, v in dashboard.report().items():
    print(f'  {k:<25}: {v}')

---

## 2 · Token Budgeting: Don't Go Bankrupt

### 🤔 Why Token Budgets?

Without budgets, one heavy user can spend your entire monthly API budget in a day.

| Budget Level | What It Controls | Example |
|-------------|-----------------|--------|
| **Per-request** | Max tokens per single request | 4,000 output tokens |
| **Per-user/day** | Daily spending limit per user | $5/day per user |
| **Per-team/month** | Monthly budget per team | $500/month for engineering |
| **Global** | Total monthly spend | $10,000/month total |

In [ ]:
# Cell 3 — Token budget enforcement
from datetime import datetime, timedelta

class TokenBudgetManager:
    """Enforce spending limits at multiple levels."""
    
    def __init__(self):
        self.limits = {}  # {entity_id: {daily: $, monthly: $}}
        self.usage = {}   # {entity_id: {today: $, month: $}}
    
    def set_limit(self, entity_id, daily=None, monthly=None):
        self.limits[entity_id] = {'daily': daily or float('inf'), 'monthly': monthly or float('inf')}
        if entity_id not in self.usage:
            self.usage[entity_id] = {'today': 0, 'month': 0}
    
    def check_budget(self, entity_id, estimated_cost):
        """Check if request is within budget BEFORE making the LLM call."""
        if entity_id not in self.limits:
            return {'allowed': True, 'reason': 'No limits set'}
        
        limit = self.limits[entity_id]
        usage = self.usage[entity_id]
        
        if usage['today'] + estimated_cost > limit['daily']:
            return {'allowed': False, 'reason': f'Daily limit exceeded (${usage["today"]:.2f} / ${limit["daily"]:.2f})',
                    'suggestion': 'Use a cheaper model or wait until tomorrow'}
        
        if usage['month'] + estimated_cost > limit['monthly']:
            return {'allowed': False, 'reason': f'Monthly limit exceeded (${usage["month"]:.2f} / ${limit["monthly"]:.2f})',
                    'suggestion': 'Contact admin to increase budget'}
        
        return {'allowed': True, 'remaining_daily': f'${limit["daily"] - usage["today"] - estimated_cost:.2f}'}
    
    def record_usage(self, entity_id, actual_cost):
        if entity_id not in self.usage:
            self.usage[entity_id] = {'today': 0, 'month': 0}
        self.usage[entity_id]['today'] += actual_cost
        self.usage[entity_id]['month'] += actual_cost

# Set up budgets
budget = TokenBudgetManager()
budget.set_limit('user_alice', daily=5.00, monthly=100.00)
budget.set_limit('user_bob', daily=2.00, monthly=50.00)
budget.set_limit('team_engineering', daily=50.00, monthly=1000.00)

# Simulate usage
print('💰 Token Budget Management')
print('=' * 60)

# Alice makes several requests
for i in range(5):
    cost = 1.20
    check = budget.check_budget('user_alice', cost)
    if check['allowed']:
        budget.record_usage('user_alice', cost)
        print(f'  ✅ Alice req {i+1}: ${cost:.2f} | Remaining: {check.get("remaining_daily", "N/A")}')
    else:
        print(f'  🔴 Alice req {i+1}: BLOCKED — {check["reason"]}')
        print(f'     Suggestion: {check["suggestion"]}')

print(f'\n  Alice total today: ${budget.usage["user_alice"]["today"]:.2f}')
print(f'\n💡 Budget checks happen BEFORE the LLM call, not after.')
print(f'   This prevents runaway costs from expensive queries.')

---

## 3 · Prompt Version Management

### 🤔 Why Version Prompts?

Changing a prompt can improve quality by 20% or break production completely. You need to:
1. Track which prompt version is being used
2. Compare quality metrics between versions
3. Roll back instantly if a new prompt causes issues

```
Prompt v1 (baseline):  Quality = 0.82, Cost = $0.015/req
Prompt v2 (improved):  Quality = 0.91, Cost = $0.018/req  ← Deploy this
Prompt v3 (bad):       Quality = 0.65, Cost = $0.020/req  ← Auto-rollback!
```

In [ ]:
# Cell 4 — Prompt version management
import hashlib, json

class PromptRegistry:
    """Track prompt versions with quality metrics."""
    
    def __init__(self):
        self.versions = []
        self.active_version = None
    
    def register(self, name, template, metadata=None):
        version_hash = hashlib.md5(template.encode()).hexdigest()[:8]
        version = {
            'name': name, 'version': len(self.versions) + 1,
            'hash': version_hash, 'template': template,
            'metrics': {'quality': 0, 'avg_cost': 0, 'requests': 0},
            'metadata': metadata or {},
        }
        self.versions.append(version)
        return version
    
    def deploy(self, version_num):
        self.active_version = self.versions[version_num - 1]
        return self.active_version
    
    def record_metrics(self, quality_score, cost):
        v = self.active_version
        v['metrics']['requests'] += 1
        n = v['metrics']['requests']
        v['metrics']['quality'] = (v['metrics']['quality'] * (n-1) + quality_score) / n
        v['metrics']['avg_cost'] = (v['metrics']['avg_cost'] * (n-1) + cost) / n
    
    def should_rollback(self, min_quality=0.8):
        v = self.active_version
        if v['metrics']['requests'] >= 10 and v['metrics']['quality'] < min_quality:
            return True
        return False

registry = PromptRegistry()
import random
random.seed(42)

# Register prompt versions
registry.register('summarizer', 'Summarize: {text}', {'author': 'alice'})
registry.register('summarizer', 'You are an expert summarizer. Summarize concisely: {text}', {'author': 'bob'})
registry.register('summarizer', 'yo just give me the tldr: {text}', {'author': 'intern'})  # Bad prompt

print('📋 Prompt Version Management')
print('=' * 60)

for v_num in [1, 2, 3]:
    v = registry.deploy(v_num)
    
    # Simulate requests with different quality
    quality_range = {1: (0.75, 0.90), 2: (0.85, 0.95), 3: (0.40, 0.70)}[v_num]
    
    for _ in range(15):
        quality = random.uniform(*quality_range)
        cost = random.uniform(0.01, 0.02)
        registry.record_metrics(quality, cost)
    
    should_rollback = registry.should_rollback(min_quality=0.8)
    status = '🔴 ROLLBACK NEEDED' if should_rollback else '🟢 OK'
    print(f'\n  v{v["version"]} [{v["hash"]}] — {status}')
    print(f'    Template: "{v["template"][:50]}..."')
    print(f'    Quality: {v["metrics"]["quality"]:.3f} | Cost: ${v["metrics"]["avg_cost"]:.4f}/req | Requests: {v["metrics"]["requests"]}')

print(f'\n💡 Prompt v3 (the intern\'s casual prompt) auto-triggered rollback.')
print(f'   In production, this would automatically revert to v2.')

---

## 🎯 Summary & Key Takeaways

| Practice | What It Solves | Tool |
|----------|---------------|------|
| Request tracing | "Why was this response slow/wrong?" | Langfuse, LangSmith |
| Token budgeting | "Who spent all our API budget?" | Custom middleware |
| Prompt versioning | "Which prompt version is better?" | Langfuse, custom registry |
| Cost dashboards | "How much are we spending per feature?" | Helicone, custom |

### 🧠 The 5 Observability Rules

1. **Trace every request** — you can't debug what you can't see
2. **Set budgets BEFORE launch** — not after the $10K surprise bill
3. **Version all prompts** — so you can compare and rollback
4. **Alert on quality drops** — not just errors
5. **Track cost per user** — identify expensive patterns

**Next →** Advanced LLM Evaluation Frameworks